In [ ]:
import json
import os
import openai
import time
import random
import pandas as pd
import pickle
from tqdm import tqdm
from helper_functions import call_api

__load API key__

In [ ]:
API_KEY = os.getenv("LLMGATEWAY_API_KEY")

__Load prompt info__

In [ ]:
with open('prompts/%s.json' % 'food') as f:
    dat = json.load(f)

__Define models__

In [ ]:
models = [
        'openai/gpt-5.5',
        'anthropic/claude-opus-4-8',
        'google-ai-studio/gemini-3.5-flash',
        #'mistral-large-latest'
        ]

__Call LLMs via API__

In [ ]:
# iterate over models 
for model in models:
    # repeat prompt N times
    N = 100

    raw_responses = [] # save full responses here
    responses = [] # save extracted text from repsonses here
    question_header = dat['question_header']
    food_options = dat['options']
    
    for i in tqdm(range(N)):
        # food option
        food_option = food_options[0]
        
        # create question_text
        question_text = dat['question_text'].format(food_option).replace('  ',' ')
        
        # call api
        response = call_api(model, question_header, question_text, API_KEY)
        answer = response.choices[0].message.content
        
        # add completion/response to list
        responses.append((question_text,model,answer,food_option))
        # save raw response to list
        raw_responses.append((response, food_option))
        
        # sleep 
        time.sleep(0.2)

    # write text reponses to file
    columns = ['question_text','model','answer','food_option']
    pd.DataFrame(responses,columns=columns).to_csv('responses/%s-%s.csv' % ('food',model.split('/')[-1]),index=False)
    
    # save raw responses to pickle file
    with open('raw_responses/%s-%s.pickle' % ('food',model.split('/')[-1]), 'wb') as outfile:
       pickle.dump(raw_responses, outfile)